In [1]:
!pip install ortools torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1

In [2]:
import os
import re
import time
import copy
import random
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from ortools.linear_solver import pywraplp
from joblib import Parallel, delayed

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv

# ==============================================================================
# DATASET LOADING & PREPROCESSING
# ==============================================================================
def load_all_datasets(folder_path):
    distance_matrices = {}
    demand_matrices = {}
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None, None

    print(f"Scanning directory: {folder_path}...\n")
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if not os.path.isfile(file_path): continue
            
        match = re.search(r'\d+', filename)
        if not match: continue
        nodes = int(match.group())
        
        try:
            if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
                df = pd.read_csv(file_path, header=None)
                distance_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes.")
                
            elif filename.startswith('wij'):
                if filename.endswith('.xlsx'): df = pd.read_excel(file_path)
                elif filename.endswith('.csv'): df = pd.read_csv(file_path)
                else: continue
                
                if df.shape[1] > nodes: df = df.iloc[:, 1:]
                demand_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes.")
        except Exception as e:
            pass
    return distance_matrices, demand_matrices

def calculate_node_production_attraction(W_matrix):
    return np.sum(W_matrix, axis=1), np.sum(W_matrix, axis=0)

def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
    np.random.seed(seed)
    n = W_matrix.shape[0]
    W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
    W_scenarios = np.maximum(W_scenarios, 0)
    return np.transpose(W_scenarios, (1, 2, 0))

def calculate_path_costs(C_matrix, alpha=0.5):
    C_ik = C_matrix[:, np.newaxis, :, np.newaxis]
    C_km = C_matrix[np.newaxis, np.newaxis, :, :]
    C_mj = C_matrix.T[np.newaxis, :, np.newaxis, :]
    return C_ik + (alpha * C_km) + C_mj

# ==============================================================================
# EXACT SOLVER & MATHEMATICAL EVALUATION
# ==============================================================================
def solve_exact_robust_model(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9, time_limit_mins=1.0, fixed_hubs=None):
    N = len(distances)
    num_scenarios = W_scenarios.shape[2]
    
    dist_scale = np.max(distances) if np.max(distances) > 0 else 1.0
    w_scale = np.max(W_scenarios) if np.max(W_scenarios) > 0 else 1.0
    scaled_distances, scaled_W = distances / dist_scale, W_scenarios / w_scale
    cost_multiplier = dist_scale * w_scale 
    
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver: return None, float('inf')
    solver.SetTimeLimit(int(time_limit_mins * 60 * 1000))

    y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(N)}
    z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for k in range(N) for i in range(N)}
    mu = solver.NumVar(0, solver.infinity(), 'mu')
    lambda_s = {s: solver.NumVar(0, solver.infinity(), f'lambda_{s}') for s in range(num_scenarios)}

    solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)

    if fixed_hubs is not None:
        for k in range(N):
            solver.Add(y[k] == (1 if k in fixed_hubs else 0))

    for i in range(N):
        solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)
        for k in range(N):
            solver.Add(z[i, k] <= y[k])

    for s in range(num_scenarios):
        W_s = scaled_W[:, :, s]
        cost_s_expr = 0
        for i in range(N):
            for k in range(N):
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[j, i] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * (alpha * scaled_distances[k, k]) * z[i, k]
        solver.Add(lambda_s[s] >= cost_s_expr - mu)

    penalty_weight = 1.0 / (num_scenarios * (1.0 - beta))
    solver.Minimize(mu + penalty_weight * solver.Sum([lambda_s[s] for s in range(num_scenarios)]))

    status = solver.Solve()
    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        return [k for k in range(N) if y[k].solution_value() > 0.5], solver.Objective().Value() * cost_multiplier
    return None, float('inf')

def evaluate_heuristic_robust_cost(distances, W_scenarios, hubs, alpha=0.5, beta=0.9):
    N, num_scenarios = len(distances), W_scenarios.shape[2]
    allocation = {i: min(hubs, key=lambda h: distances[i, h]) for i in range(N)}
    
    path_costs = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            hi, hj = allocation[i], allocation[j]
            path_costs[i, j] = distances[i, hi] + alpha * distances[hi, hj] + distances[hj, j]
            
    scenario_costs = np.tensordot(path_costs, W_scenarios, axes=([0, 1], [0, 1]))
    sorted_costs = np.sort(scenario_costs)
    var_index = int(beta * num_scenarios)
    mu = sorted_costs[var_index]
    
    return mu + np.sum(sorted_costs[var_index:] - mu) / (num_scenarios * (1.0 - beta))

# ==============================================================================
# GRAPH NEURAL NETWORK & HYPERPARAMETER TUNING
# ==============================================================================
def create_multi_graph_data(distances, demands, labels=None):
    N = len(distances)
    production, attraction = calculate_node_production_attraction(demands)
    spatial_centrality = 1.0 / (np.sum(distances, axis=1) + 1e-6)
    
    x = torch.tensor(np.column_stack((
        production / (np.max(production) or 1.0), 
        attraction / (np.max(attraction) or 1.0),
        spatial_centrality / (np.max(spatial_centrality) or 1.0)
    )), dtype=torch.float)

    edge_index, edge_attr = [], []
    dist_max, dem_max = (np.max(distances) or 1.0), (np.max(demands) or 1.0)
    flow = demands + demands.T
    flow_max = np.max(flow) or 1.0

    for i in range(N):
        for j in range(N):
            if i != j:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]/dist_max, demands[i, j]/dem_max, flow[i, j]/flow_max])

    data = Data(x=x, 
                edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
                edge_attr=torch.tensor(edge_attr, dtype=torch.float))
    if labels is not None: data.y = torch.tensor(labels, dtype=torch.float)
    return data

class MultiGraphHubRanker(torch.nn.Module):
    def __init__(self, node_in_dim=3, edge_in_dim=3, hidden_dim=64):
        super(MultiGraphHubRanker, self).__init__()
        self.conv1 = TransformerConv(node_in_dim, hidden_dim, edge_dim=edge_in_dim)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim, edge_dim=edge_in_dim)
        self.fc1 = torch.nn.Linear(hidden_dim, 32)
        self.dropout = torch.nn.Dropout(p=0.2)
        self.fc2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x = self.dropout(F.relu(self.conv1(data.x, data.edge_index, data.edge_attr)))
        x = self.dropout(F.relu(self.conv2(x, data.edge_index, data.edge_attr)))
        return self.fc2(F.relu(self.fc1(x))).squeeze()

def train_dl_model(model, train_data_list, val_data_list=None, epochs=300, lr=0.01, patience=20):
    """
    [HYPERPARAMETER SEARCH UPDATE]: Now explicitly supports a Validation Set.
    Early stopping and best-model tracking are entirely based on Validation Loss
    to guarantee the model does not overfit to the training set.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Calculate pos_weight based on the training set
    all_labels = torch.cat([data.y for data in train_data_list])
    pos_weight = ((len(all_labels) - all_labels.sum()) / (all_labels.sum() + 1e-5)).clone().detach().to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    # If no validation set provided, fallback to using train set for early stopping
    eval_data_list = val_data_list if val_data_list is not None else train_data_list
    
    best_loss, epochs_no_improve = float('inf'), 0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        for data in train_data_list:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y)
            loss.backward()
            optimizer.step()
            
        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in eval_data_list:
                data = data.to(device)
                loss = criterion(model(data), data.y)
                total_val_loss += loss.item()
                
        avg_val_loss = total_val_loss / len(eval_data_list)
        
        if avg_val_loss < best_loss - 1e-4:
            best_loss, epochs_no_improve, best_model_wts = avg_val_loss, 0, copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            
        if (epoch + 1) % 10 == 0 or epochs_no_improve >= patience:
            print(f"   Epoch {epoch+1:03d}/{epochs} | Val Loss: {avg_val_loss:.4f} | Best Val: {best_loss:.4f} | Patience: {epochs_no_improve}/{patience}")
            
        if epochs_no_improve >= patience:
            print(f"   [Early Stopping] Triggered at epoch {epoch+1}!")
            break
            
    print(f"   [Complete] Restored weights at Val Loss: {best_loss:.4f}")
    model.load_state_dict(best_model_wts)
    return model, best_loss

def generate_subsampled_training_data(real_distances, real_demands, target_N):
    N = len(real_distances)
    if target_N >= N: return real_distances, real_demands
    indices = np.random.choice(N, target_N, replace=False)
    return real_distances[indices][:, indices], real_demands[indices][:, indices]

# ==============================================================================
# ALGORITHMS (BASELINES & AI-AUGMENTED) WITH PARALLELIZATION
# ==============================================================================

def run_cbs(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0) 
    return _cbs_core(distances, W_scenarios, naive_rankings, p_hubs, alpha, beta, num_clusters)

def run_dl_cbs(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    return _cbs_core(distances, W_scenarios, dl_rankings, p_hubs, alpha, beta, num_clusters)

def _eval_combo_worker(combo, distances, W_scenarios, alpha, beta):
    cost = evaluate_heuristic_robust_cost(distances, W_scenarios, list(combo), alpha, beta)
    return cost, list(combo)

def _cbs_core(distances, W_scenarios, rankings, p_hubs, alpha, beta, num_clusters, top_c=3):
    N = len(distances)
    num_clusters = max(1, min(num_clusters, N // top_c))
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(distances) 
    
    candidate_hubs = set()
    for cluster_idx in range(num_clusters):
        nodes = np.where(cluster_labels == cluster_idx)[0]
        sorted_nodes = sorted(nodes, key=lambda x: rankings[x], reverse=True)
        candidate_hubs.update(sorted_nodes[:top_c])
        
    candidate_hubs.update(np.argsort(rankings)[-min(N, p_hubs * 2):]) 
    candidate_hubs = list(candidate_hubs)
    
    if len(candidate_hubs) > 16:
        candidate_hubs = sorted(candidate_hubs, key=lambda x: rankings[x], reverse=True)[:16]
        
    if len(candidate_hubs) < p_hubs: 
        candidate_hubs = list(np.argsort(rankings)[-p_hubs:])

    combos = list(itertools.combinations(candidate_hubs, p_hubs))
    
    results = Parallel(n_jobs=-1)(
        delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in combos
    )
    
    if results:
        best_cost, best_hubs = min(results, key=lambda x: x[0])
        return best_hubs, best_cost
    return None, float('inf')

def run_gvns(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, max_iter=50):
    N = len(distances)
    naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0)
    candidate_space = np.argsort(naive_rankings)[-min(N, 40):] 
    current_hubs = list(np.random.choice(candidate_space, p_hubs, replace=False))
    return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)

def run_dl_gvns(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, max_iter=50):
    N = len(distances)
    current_hubs = list(np.argsort(dl_rankings)[-p_hubs:])
    candidate_space = np.argsort(dl_rankings)[-min(N, 40):]
    return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)

def _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter):
    best_hubs = initial_hubs.copy()
    best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    
    for _ in range(max_iter):
        improved = True
        while improved:
            improved = False
            for i in range(len(best_hubs)):
                for c in candidate_space:
                    if c not in best_hubs:
                        test_hubs = best_hubs.copy()
                        test_hubs[i] = c
                        cost = evaluate_heuristic_robust_cost(distances, W_scenarios, test_hubs, alpha, beta)
                        if cost < best_cost:
                            best_cost, best_hubs, improved = cost, test_hubs, True
                            
        shake_idx = np.random.randint(0, len(best_hubs))
        shake_candidate = np.random.choice([c for c in candidate_space if c not in best_hubs])
        best_hubs[shake_idx] = shake_candidate
        
    best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    return best_hubs, best_cost


# ==============================================================================
# PHASE 4: RESULTS EXPORT & PLOTTING
# ==============================================================================
def plot_academic_results(df):
    print("\n[Plotting] Generating Academic Charts...")
    sns.set_theme(style="whitegrid")
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Nodes', y='Robust Cost', hue='Method')
    plt.title("Robust Cost Comparison Across Methods")
    plt.ylabel("Conditional Value at Risk (Cost)")
    plt.yscale("log") 
    plt.tight_layout()
    plt.savefig("Fig1_Cost_Comparison.png", dpi=300)
    plt.close()

    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Nodes', y='Time (s)', hue='Method')
    plt.title("Execution Time by Method and Network Size")
    plt.ylabel("Time (seconds)")
    plt.yscale("log") 
    plt.tight_layout()
    plt.savefig("Fig2_Time_Comparison.png", dpi=300)
    plt.close()

    df_heuristics = df[df['Method'] != 'Exact (SCIP)']
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_heuristics, x='Nodes', y='Gap (%)', hue='Method', marker='o', linewidth=2.5)
    plt.title("Optimality Gap Scaling with Network Size")
    plt.ylabel("Gap from Exact Solver (%)")
    plt.axhline(0, color='black', linestyle='--')
    plt.tight_layout()
    plt.savefig("Fig3_Gap_Scaling.png", dpi=300)
    plt.close()
    
    print("[Plotting] Charts saved: Fig1_Cost_Comparison.png, Fig2_Time_Comparison.png, Fig3_Gap_Scaling.png")

# ==============================================================================
# PARALLEL WORKER FOR DATA GENERATION
# ==============================================================================
def _generate_training_graph_worker(i, common_nodes, distance_matrices, demand_matrices):
    np.random.seed((int(time.time() * 1000) + i) % 123456789)
    base_N = np.random.choice(common_nodes)
    sub_N = min(np.random.randint(12, 23), base_N) 
    train_dist, train_dem = generate_subsampled_training_data(distance_matrices[base_N], demand_matrices[base_N], sub_N)
    
    W_scen = generate_demand_scenarios(train_dem, num_scenarios=15, seed=np.random.randint(10000)) 
    
    try:
        exact_hubs, _ = solve_exact_robust_model(train_dist, W_scen, p_hubs=3, time_limit_mins=0.5)
        if exact_hubs:
            labels = np.zeros(sub_N)
            labels[exact_hubs] = 1.0
            return create_multi_graph_data(train_dist, train_dem, labels=labels)
    except Exception:
        pass 
    return None

# ==============================================================================
# MASTER PIPELINE EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print("==================================================")
    print("🚀 STARTING PARALLELIZED ACADEMIC PIPELINE")
    print("==================================================")
    
    DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
    distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
    if distance_matrices and demand_matrices:
        common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
        
        # --- PHASE 1: GENERATE TRAINING DATA (PARALLEL) ---
        num_training_graphs = 200 
        print(f"\n[Phase 1] Generating {num_training_graphs} Realistic Training Sub-graphs (Using Multi-Core Processing)...")
        
        parallel_results = Parallel(n_jobs=-1)(
            delayed(_generate_training_graph_worker)(i, common_nodes, distance_matrices, demand_matrices) 
            for i in range(num_training_graphs)
        )
        
        training_graphs = [g for g in parallel_results if g is not None]
        print(f"   [✓] Successfully generated {len(training_graphs)}/{num_training_graphs} graphs.")
            
        # --- PHASE 2: HYPERPARAMETER SEARCH (GNN) ---
        print("\n\n" + "="*50)
        print("PHASE 2: GNN HYPERPARAMETER SEARCH & TRAINING")
        print("="*50)
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        best_model = None
        best_val_loss = float('inf')
        best_hidden_dim = None
        
        if len(training_graphs) > 4:
            # 80/20 Train-Validation Split
            random.shuffle(training_graphs)
            split_idx = int(len(training_graphs) * 0.8)
            train_set = training_graphs[:split_idx]
            val_set = training_graphs[split_idx:]
            
            # The Architectures to evaluate (Neurons per hidden layer)
            hidden_dims_to_test = [32, 64, 128]
            
            print(f"Train Set: {len(train_set)} graphs | Validation Set: {len(val_set)} graphs")
            
            for h_dim in hidden_dims_to_test:
                print(f"\n--- Testing Architecture: Hidden Dimension = {h_dim} ---")
                temp_model = MultiGraphHubRanker(node_in_dim=3, edge_in_dim=3, hidden_dim=h_dim)
                
                trained_model, final_val_loss = train_dl_model(
                    temp_model, train_data_list=train_set, val_data_list=val_set, 
                    epochs=300, patience=20
                )
                
                if final_val_loss < best_val_loss:
                    best_val_loss = final_val_loss
                    best_model = trained_model
                    best_hidden_dim = h_dim
                    
            print(f"\n🌟 [Hyperparameter Search Complete]")
            print(f"   Winning Architecture: {best_hidden_dim} Neurons (Val Loss: {best_val_loss:.4f})")
            dl_model = best_model
            dl_model.eval()
        else:
            print("[!] Not enough training data generated. Skipping Phase 2.")
            dl_model = MultiGraphHubRanker()
        
        # --- PHASE 3: PARAMETER GRID SEARCH ---
        print("\n\n" + "="*50)
        print("PHASE 3: GRID SEARCH & BASELINE EVALUATIONS")
        print("="*50)
        
        results_list = []
        p_values = [2, 3, 4] 
        
        for N in common_nodes:
            distances, demands = distance_matrices[N], demand_matrices[N]
            W_scenarios = generate_demand_scenarios(demands, num_scenarios=100)
            
            with torch.no_grad():
                graph_data = create_multi_graph_data(distances, demands).to(device)
                ai_probs = torch.sigmoid(dl_model(graph_data)).cpu().numpy()
            
            for p in p_values:
                print(f"\n--- Evaluating Nodes={N} | Hubs(p)={p} ---")
                
                # 1. Exact Solver
                start_t = time.time()
                try:
                    exact_hubs, exact_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=1.5)
                except:
                    exact_cost = float('inf')
                t_exact = time.time() - start_t
                
                if exact_cost != float('inf'):
                    results_list.append({'Nodes': N, 'Hubs': p, 'Method': 'Exact (SCIP)', 'Time (s)': t_exact, 'Robust Cost': exact_cost, 'Gap (%)': 0.0})
                    print(f"   [Exact] Time: {t_exact:.2f}s | Cost: {exact_cost:.2e}")
                
                def run_and_log(method_name, func, *args):
                    start_t = time.time()
                    hubs, cost = func(*args)
                    
                    if exact_cost != float('inf'):
                        _, true_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=0.5, fixed_hubs=hubs)
                        gap = ((true_cost - exact_cost) / exact_cost) * 100
                    else:
                        true_cost, gap = cost, 0.0
                        
                    t_val = time.time() - start_t
                    results_list.append({'Nodes': N, 'Hubs': p, 'Method': method_name, 'Time (s)': t_val, 'Robust Cost': true_cost, 'Gap (%)': gap})
                    print(f"   [{method_name}] Time: {t_val:.2f}s | Gap: {gap:.2f}% | Hubs: {hubs}")

                # 2. Heuristics Evaluation
                run_and_log('CBS (Standard)', run_cbs, distances, W_scenarios, demands, p)
                run_and_log('DL-CBS (AI)', run_dl_cbs, distances, W_scenarios, ai_probs, p)
                run_and_log('GVNS (Standard)', run_gvns, distances, W_scenarios, demands, p)
                run_and_log('DL-GVNS (AI)', run_dl_gvns, distances, W_scenarios, ai_probs, p)

        # --- PHASE 4: EXPORT & PLOT ---
        if results_list:
            df_results = pd.DataFrame(results_list)
            df_results.to_csv("All_Datasets_Results.csv", index=False)
            plot_academic_results(df_results)
            
            print("\n==================================================")
            print("✅ BATCH PIPELINE COMPLETE!")
            print("📈 All plots saved as PNG images.")
            print("💾 Master data saved to 'All_Datasets_Results.csv'.")
            print("==================================================")

    else:
        print("\n[!] Directory not found or empty.")

🚀 STARTING PARALLELIZED ACADEMIC PIPELINE
Scanning directory: /kaggle/input/datasets/infernalss/node-dataset...

[✓] Loaded Demand Matrix (W) for 100 nodes.
[✓] Loaded Distance Matrix (C) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 100 nodes.
[✓] Loaded Demand Matrix (W) for 25 nodes.
[✓] Loaded Distance Matrix (C) for 25 nodes.

[Phase 1] Generating 200 Realistic Training Sub-graphs (Using Multi-Core Processing)...
   [✓] Successfully generated 200/200 graphs.


PHASE 2: GNN HYPERPARAMETER SEARCH & TRAINING
Train Set: 160 graphs | Validation Set: 40 graphs

--- Testing Architecture: Hidden Dimension = 32 ---
   Epoch 010/300 | Val Loss: 0.7215 | Best Val: 0.6950 | Patience: 1/20
   Epoch 020/300 | Val Loss: 0.6886 | Best Val: 0.6837 | Patience: 1/20
   Epoch 030/300 | Val Loss: 0.6814 | Best Val: 0.6814 | Patience: 0/20
   Epoch 040/300 | Val Loss